In [ ]:
# Common imports
import numpy as np
import pandas as pd
from termcolor import colored
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
%config InlineBackend.figure_format = 'retina'
sns.set_style("darkgrid")
plt.rcParams['figure.figsize'] = [10, 6]

print(colored('Imports loaded successfully ✓', 'green'))

## 1. Data Loading

In [ ]:
# Load datasets
df_games = pd.read_csv('../data/games_preprocessed.csv')
df_interactions = pd.read_csv('../data/dataset_interactions.csv')

# Load all vectorization vectors
with open('../data/vectors_tfidf.pkl', 'rb') as f:
    vectors_tfidf = pickle.load(f)
with open('../data/vectors_w2v.pkl', 'rb') as f:
    vectors_w2v = pickle.load(f)
with open('../data/vectors_lda.pkl', 'rb') as f:
    vectors_lda = pickle.load(f)
with open('../data/vectors_bert.pkl', 'rb') as f:
    vectors_bert = pickle.load(f)

print(colored(f'Games: {len(df_games)}', 'green'))
print(colored(f'Interactions: {len(df_interactions)} | Players: {df_interactions["player_id"].nunique()} | Games: {df_interactions["app_id"].nunique()}', 'green'))
print(colored(f'TF-IDF vectors: {len(vectors_tfidf)} docs', 'blue'))
print(colored(f'W2V vectors: {vectors_w2v.shape}', 'blue'))
print(colored(f'LDA vectors: {vectors_lda.shape}', 'blue'))
print(colored(f'BERT vectors: {vectors_bert.shape}', 'blue'))

## 2. Data Preprocessing

In [ ]:
from sklearn.preprocessing import LabelEncoder
from gensim.matutils import corpus2csc

# Encode player_id and app_id as integers
enc_players = LabelEncoder()
enc_items = LabelEncoder()

df_interactions['player_enc'] = enc_players.fit_transform(df_interactions['player_id'])
df_interactions['item_enc'] = enc_items.fit_transform(df_interactions['app_id'])

num_users = df_interactions['player_enc'].nunique()
num_items = df_interactions['item_enc'].nunique()

print(colored(f'Interactions after filtering: {len(df_interactions)}', 'green'))
print(colored(f'Unique players: {num_users} | Unique games: {num_items}', 'green'))

# Build app_id -> index mapping for vectors
app_id_to_idx = {app_id: idx for idx, app_id in enumerate(df_games['app_id'].values)}

# Build item_enc -> vector index mapping
encoded_app_ids = enc_items.classes_  # app_ids in encoded order
item_enc_to_vec_idx = {
    i: app_id_to_idx[app_id]
    for i, app_id in enumerate(encoded_app_ids)
    if app_id in app_id_to_idx
}

print(colored(f'Items with vector mapping: {len(item_enc_to_vec_idx)}', 'green'))

# Build TF-IDF dense matrix (536 x vocab)
from gensim.matutils import corpus2csc
tfidf_matrix = corpus2csc(vectors_tfidf, num_terms=None).T.toarray()
print(colored(f'TF-IDF matrix shape: {tfidf_matrix.shape}', 'blue'))

## 3. Content-Based Filtering

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def content_based_recommender(app_id, vectors, df_games, top_n=10):
    """
    Recommends games similar to the given app_id based on cosine similarity
    of the provided vector representation.
    """
    if app_id not in app_id_to_idx:
        print(colored(f'Game not found: {app_id}', 'red'))
        return None
    
    idx = app_id_to_idx[app_id]
    sim_scores = cosine_similarity([vectors[idx]], vectors)[0]
    similar_indices = sim_scores.argsort()[::-1][1:top_n+1]
    
    results = df_games.iloc[similar_indices][['title', 'genre', 'rating']].copy()
    results['similarity'] = sim_scores[similar_indices]
    results = results.reset_index(drop=True)
    return results

# Test with all 4 vectorizations
test_game = 'com.supercell.clashofclans'
test_title = df_games[df_games['app_id'] == test_game]['title'].values[0]

print(colored(f'Recommendations for: {test_title}', 'blue'))
print()

for name, vectors in [('TF-IDF', tfidf_matrix), 
                       ('Word2Vec', vectors_w2v),
                       ('LDA', vectors_lda), 
                       ('BERT', vectors_bert)]:
    print(colored(f'--- {name} ---', 'red'))
    recs = content_based_recommender(test_game, vectors, df_games, top_n=5)
    if recs is not None:
        print(recs.to_string(index=False))
    print()

## 3.1 Quantitative Comparison of Vectorizations

Before analyzing individual recommendations, we evaluate the semantic quality of each vectorization method by measuring **cosine similarity within and across genres**.

The intuition is straightforward: a good document representation for content-based recommendation should assign **higher similarity to games of the same genre** than to games of different genres. The gap between within-genre and across-genre similarity serves as a proxy for how well each vectorization captures genre-level semantics.

We compute this metric for all four representations:
- **TF-IDF**: sparse bag-of-words weighted by inverse document frequency
- **Word2Vec**: TF-IDF weighted average of 100-dimensional word embeddings
- **LDA**: 20-dimensional topic distribution vector
- **BERT**: 384-dimensional contextual sentence embeddings (`paraphrase-multilingual-MiniLM-L12-v2`)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity as cos_sim
import numpy as np

print(colored('Computing within/across-genre similarity for all vectorizations...', 'blue'))

genres = df_games['genre'].values

def genre_similarity_analysis(vectors, name):
    """Compute avg within-genre and across-genre cosine similarity."""
    sim_matrix = cos_sim(vectors)
    within, across = [], []
    for i in range(len(df_games)):
        for j in range(i+1, len(df_games)):
            if genres[i] == genres[j]:
                within.append(sim_matrix[i, j])
            else:
                across.append(sim_matrix[i, j])
    print(colored(
        f'{name:<10} | Within-genre: {np.mean(within):.4f} | '
        f'Across-genre: {np.mean(across):.4f} | '
        f'Gap: {np.mean(within) - np.mean(across):.4f}',
        'green'
    ))
    return np.mean(within), np.mean(across)

# Run for all 4 vectorizations
results_sim = {}
results_sim['TF-IDF']   = genre_similarity_analysis(tfidf_matrix, 'TF-IDF')
results_sim['Word2Vec'] = genre_similarity_analysis(vectors_w2v, 'Word2Vec')
results_sim['LDA']      = genre_similarity_analysis(vectors_lda, 'LDA')
results_sim['BERT']     = genre_similarity_analysis(vectors_bert, 'BERT')

# Bar chart comparison
fig, ax = plt.subplots(figsize=(10, 5))
names = list(results_sim.keys())
within_vals = [results_sim[n][0] for n in names]
across_vals = [results_sim[n][1] for n in names]

x = np.arange(len(names))
width = 0.35
bars1 = ax.bar(x - width/2, within_vals, width, label='Within-genre', color='steelblue')
bars2 = ax.bar(x + width/2, across_vals, width, label='Across-genre', color='coral')

ax.set_title('Avg Cosine Similarity by Vectorization: Within vs Across Genre')
ax.set_xlabel('Vectorization method')
ax.set_ylabel('Average Cosine Similarity')
ax.set_xticks(x)
ax.set_xticklabels(names)
ax.legend()
ax.grid(axis='y')

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

print(colored('\nConclusion: higher within-genre gap = better semantic discrimination by genre', 'blue'))

### Results

| Method   | Within-genre | Across-genre | Gap    |
|----------|-------------|--------------|--------|
| TF-IDF   | 0.053       | 0.020        | 0.033  |
| Word2Vec | 1.000       | 1.000        | 0.000  |
| LDA      | 0.212       | 0.114        | 0.098  |
| BERT     | 0.371       | 0.245        | 0.126  |

**BERT** achieves the highest within-genre gap (0.126), meaning it best captures genre-level semantic similarity. **LDA** ranks second (0.098), benefiting from its explicit topic structure. **TF-IDF** shows low absolute similarity values (sparse representation) but a meaningful relative gap. **Word2Vec** shows zero discrimination (gap = 0.000) on this corpus, so all document vectors collapse to near-identical representations, confirming that averaging word embeddings on a small, domain-homogeneous dataset produces degenerate document vectors.

These results are consistent with the qualitative analysis in Section 3: BERT and TF-IDF produce the most genre-coherent recommendations for Clash of Clans, while Word2Vec recommendations are essentially random with respect to genre.

## 4. Collaborative Filtering

In [ ]:
from sklearn.model_selection import train_test_split
from surprise import Dataset, Reader, KNNBasic, SVD
from surprise.model_selection import cross_validate, GridSearchCV as SurpriseGridSearch
from surprise import accuracy

try:
    from surprise import Dataset
except:
    import subprocess
    subprocess.run(['pip', 'install', 'scikit-surprise'])
    from surprise import Dataset, Reader, KNNBasic, SVD

# Prepare data for Surprise
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(
    df_interactions[['player_id', 'app_id', 'rating']],
    reader
)

# Train/test split
from surprise.model_selection import train_test_split as surprise_split
trainset, testset = surprise_split(data, test_size=0.2, random_state=42)

print(colored(f'Trainset size: {trainset.n_ratings}', 'green'))
print(colored(f'Testset size: {len(testset)}', 'green'))

# ── Hyperparameter validation for KNN ──────────────────────────────────────
print(colored('\nRunning KNN cross-validation...', 'blue'))

param_grid_knn = {
    'k': [10, 20, 40, 60],
    'sim_options': {
        'name': ['cosine', 'pearson'],
        'user_based': [True]        # user-based search
    }
}

gs_knn_user = SurpriseGridSearch(
    KNNBasic, param_grid_knn, measures=['rmse'], cv=3, n_jobs=-1
)
gs_knn_user.fit(data)

print(colored(f'KNN User-based — best RMSE: {gs_knn_user.best_score["rmse"]:.4f}', 'green'))
print(colored(f'KNN User-based — best params: {gs_knn_user.best_params["rmse"]}', 'green'))

# Item-based search
param_grid_knn_item = {
    'k': [10, 20, 40, 60],
    'sim_options': {
        'name': ['cosine', 'pearson'],
        'user_based': [False]       # item-based search
    }
}

gs_knn_item = SurpriseGridSearch(
    KNNBasic, param_grid_knn_item, measures=['rmse'], cv=3, n_jobs=-1
)
gs_knn_item.fit(data)

print(colored(f'KNN Item-based — best RMSE: {gs_knn_item.best_score["rmse"]:.4f}', 'green'))
print(colored(f'KNN Item-based — best params: {gs_knn_item.best_params["rmse"]}', 'green'))

# ── Train final KNN models with optimal parameters ─────────────────────────
best_params_user = gs_knn_user.best_params['rmse']
best_params_item = gs_knn_item.best_params['rmse']

print(colored('\nTraining KNN User-based (optimized)...', 'blue'))
knn_user = KNNBasic(
    k=best_params_user['k'],
    sim_options=best_params_user['sim_options']
)
knn_user.fit(trainset)
preds_knn_user = knn_user.test(testset)
rmse_knn_user = accuracy.rmse(preds_knn_user, verbose=False)

print(colored('Training KNN Item-based (optimized)...', 'blue'))
knn_item = KNNBasic(
    k=best_params_item['k'],
    sim_options=best_params_item['sim_options']
)
knn_item.fit(trainset)
preds_knn_item = knn_item.test(testset)
rmse_knn_item = accuracy.rmse(preds_knn_item, verbose=False)

# ── SVD baseline (default, will be optimized in section 9) ─────────────────
print(colored('Training SVD (Matrix Factorization)...', 'blue'))
svd = SVD(n_factors=50, n_epochs=20, random_state=42)
svd.fit(trainset)
preds_svd = svd.test(testset)
rmse_svd = accuracy.rmse(preds_svd, verbose=False)

print(colored('\n============= Collaborative Filtering Results =============', 'blue'))
print(colored(f'KNN User-based RMSE: {rmse_knn_user:.4f}  (k={best_params_user["k"]}, sim={best_params_user["sim_options"]["name"]})', 'green'))
print(colored(f'KNN Item-based RMSE: {rmse_knn_item:.4f}  (k={best_params_item["k"]}, sim={best_params_item["sim_options"]["name"]})', 'green'))
print(colored(f'SVD RMSE:            {rmse_svd:.4f}', 'green'))

## 4.1 Unified Evaluation: HR@10 for Collaborative Filtering Models

To enable a fair comparison between all recommender models, we evaluate KNN and SVD using **Hit Rate @ 10 (HR@10)** — the same metric used for NCF and Hybrid NCF.

**Protocol**: for each user, we hold out their last rated item as the positive sample and rank it against 99 randomly sampled negative items (items the user has not interacted with). HR@10 measures the fraction of users where the positive item appears in the top-10 ranked candidates.

In [ ]:
# HR@10 evaluation for Surprise models (KNN and SVD)
# Same protocol as NCF: for each user, rank the held-out item
# against 99 random negatives and check if it appears in top-10

# Build train/test user-item dicts if not already in memory
# (these are also built in Section 5 for NCF evaluation)
if 'train_user_items_dict' not in globals():
    train_user_items_dict = {}
    test_user_items_dict = {}
    for _, row in df_interactions.iterrows():
        u = int(row['player_enc'])
        i = int(row['item_enc'])
        train_user_items_dict.setdefault(u, set()).add(i)
    test_users = list(train_user_items_dict.keys())[-int(len(train_user_items_dict)*0.2):]
    for u in test_users:
        items = list(train_user_items_dict[u])
        if len(items) > 1:
            test_user_items_dict[u] = {items[-1]}
            train_user_items_dict[u] = set(items[:-1])

def hit_rate_surprise(model, train_user_items, test_user_items,
                      enc_players, enc_items, num_negatives=99, k=10):
    """
    Compute HR@10 for a Surprise model using the same protocol as NCF.
    For each user, rank the held-out positive item against num_negatives
    random negative items using the model's predicted rating.
    """
    hits = 0
    total = 0
    all_item_ids_set = set(range(num_items))

    for user_enc, pos_items in test_user_items.items():
        # Get original player_id for Surprise
        try:
            player_id = enc_players.inverse_transform([user_enc])[0]
        except:
            continue

        seen_items = train_user_items.get(user_enc, set())

        for pos_item_enc in pos_items:
            try:
                pos_app_id = enc_items.inverse_transform([pos_item_enc])[0]
            except:
                continue

            # Sample negatives
            available = list(all_item_ids_set - seen_items - {pos_item_enc})
            if len(available) < num_negatives:
                continue
            neg_item_encs = np.random.choice(available, num_negatives, replace=False)

            # Build candidate list: 1 positive + 99 negatives
            candidates_enc = [pos_item_enc] + list(neg_item_encs)
            candidates_app_ids = enc_items.inverse_transform(candidates_enc)

            # Get predicted ratings from Surprise model
            scores = [
                model.predict(player_id, app_id).est
                for app_id in candidates_app_ids
            ]

            # Check if positive item is in top-k
            top_k_indices = np.argsort(scores)[::-1][:k]
            if 0 in top_k_indices:  # positive item is at index 0
                hits += 1
            total += 1

    return hits / total if total > 0 else 0.0

print(colored('Evaluating KNN and SVD with HR@10...', 'blue'))
print(colored('(This may take a few minutes)', 'blue'))

np.random.seed(42)
hr10_knn_user = hit_rate_surprise(knn_user, train_user_items_dict,
                                   test_user_items_dict, enc_players, enc_items)
print(colored(f'KNN User-based HR@10: {hr10_knn_user:.4f}', 'green'))

hr10_knn_item = hit_rate_surprise(knn_item, train_user_items_dict,
                                   test_user_items_dict, enc_players, enc_items)
print(colored(f'KNN Item-based HR@10: {hr10_knn_item:.4f}', 'green'))

hr10_svd = hit_rate_surprise(svd, train_user_items_dict,
                              test_user_items_dict, enc_players, enc_items)
print(colored(f'SVD HR@10:  {hr10_svd:.4f}', 'green'))

## 5. Deep Collaborative Filtering (Neural CF)

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset as TorchDataset, DataLoader
import pytorch_lightning as pl

# Prepare implicit feedback dataset
# Convert to binary: 1 if interacted, generate negatives
train_df = df_interactions.copy()

# Build user-item interaction sets
train_user_items = (
    train_df.groupby('player_enc')['item_enc']
    .apply(set).to_dict()
)
all_item_ids = list(range(num_items))

class ImplicitTrainDataset(TorchDataset):
    def __init__(self, ratings_df, all_item_ids, num_negatives=4):
        self.ratings_df = ratings_df
        self.all_item_ids = all_item_ids
        self.num_negatives = num_negatives
        self.user_items = (
            ratings_df.groupby('player_enc')['item_enc']
            .apply(set).to_dict()
        )
        self.samples = self._generate_samples()

    def _generate_samples(self):
        samples = []
        for _, row in self.ratings_df.iterrows():
            u, i = int(row['player_enc']), int(row['item_enc'])
            samples.append((u, i, 1.0))
            seen = self.user_items.get(u, set())
            neg_count = 0
            while neg_count < self.num_negatives:
                neg = np.random.choice(self.all_item_ids)
                if neg not in seen:
                    samples.append((u, neg, 0.0))
                    neg_count += 1
        return samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        u, i, label = self.samples[idx]
        return torch.tensor(u, dtype=torch.long), torch.tensor(i, dtype=torch.long), torch.tensor(label, dtype=torch.float)

print(colored('Dataset classes defined ✓', 'green'))

In [ ]:
class NCF(pl.LightningModule):
    def __init__(self, num_users, num_items, ratings_df, all_item_ids,
                 user_embedding_dim=16, item_embedding_dim=16,
                 hidden_dim=64, num_negatives=4, lr=1e-3):
        super().__init__()
        self.save_hyperparameters(ignore=['ratings_df', 'all_item_ids'])
        self.user_embedding = nn.Embedding(num_users, user_embedding_dim)
        self.item_embedding = nn.Embedding(num_items, item_embedding_dim)
        self.fc1 = nn.Linear(user_embedding_dim + item_embedding_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, 32)
        self.output = nn.Linear(32, 1)
        self.ratings_df = ratings_df
        self.all_item_ids = all_item_ids
        self.num_negatives = num_negatives
        self.lr = lr

    def forward(self, user_input, item_input):
        user_emb = self.user_embedding(user_input)
        item_emb = self.item_embedding(item_input)
        x = torch.cat([user_emb, item_emb], dim=-1)
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return torch.sigmoid(self.output(x))

    def training_step(self, batch, batch_idx):
        u, i, labels = batch
        preds = self(u, i).view(-1)
        loss = nn.BCELoss()(preds, labels)
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.lr)

    def train_dataloader(self):
        dataset = ImplicitTrainDataset(self.ratings_df, self.all_item_ids, self.num_negatives)
        return DataLoader(dataset, batch_size=512, shuffle=True, num_workers=0)

print(colored('NCF model defined ✓', 'green'))
print(colored('Training NCF...', 'blue'))

model_ncf = NCF(
    num_users=num_users,
    num_items=num_items,
    ratings_df=train_df,
    all_item_ids=all_item_ids,
    user_embedding_dim=16,
    item_embedding_dim=16,
    hidden_dim=64,
    num_negatives=4,
    lr=1e-3
)

trainer = pl.Trainer(max_epochs=5, enable_checkpointing=False, logger=False)
trainer.fit(model_ncf)
print(colored('NCF trained ✓', 'green'))

### 5.1 NCF Evaluation (Hit Rate)

In [ ]:
def hit_rate_at_k(model, train_user_items, test_user_items, all_item_ids, k=10, num_negatives=99):
    model.eval()
    hits = 0
    total = 0

    for user_id, pos_items in test_user_items.items():
        seen_items = train_user_items.get(user_id, set())
        
        for pos_item in pos_items:
            # Sample negatives
            candidates = [pos_item]
            neg_count = 0
            while neg_count < num_negatives:
                neg = np.random.choice(all_item_ids)
                if neg not in seen_items and neg != pos_item:
                    candidates.append(neg)
                    neg_count += 1

            user_tensor = torch.tensor([user_id] * len(candidates), dtype=torch.long)
            item_tensor = torch.tensor(candidates, dtype=torch.long)

            with torch.no_grad():
                scores = model(user_tensor, item_tensor).view(-1).cpu().numpy()

            top_k_items = np.array(candidates)[np.argsort(scores)[::-1][:k]]
            if pos_item in top_k_items:
                hits += 1
            total += 1

    return hits / total if total > 0 else 0.0

# Build test user items from testset
test_user_items_dict = {}
train_user_items_dict = {}

for _, row in df_interactions.iterrows():
    u = int(row['player_enc'])
    i = int(row['item_enc'])
    train_user_items_dict.setdefault(u, set()).add(i)

# Use last 20% users for test
test_users = list(train_user_items_dict.keys())[-int(len(train_user_items_dict)*0.2):]
for u in test_users:
    items = list(train_user_items_dict[u])
    if len(items) > 1:
        test_user_items_dict[u] = {items[-1]}
        train_user_items_dict[u] = set(items[:-1])

print(colored('Evaluating NCF...', 'blue'))
hr10_ncf = hit_rate_at_k(model_ncf, train_user_items_dict, test_user_items_dict, all_item_ids, k=10)
print(colored(f'NCF HR@10: {hr10_ncf:.4f}', 'green'))

## 6. Hybrid Deep Recommender (NCF + BERT embeddings)

In [ ]:
# Prepare BERT title embedding matrix aligned with encoded item IDs
bert_embedding_dim = vectors_bert.shape[1]  # 384
desc_embedding_matrix = np.zeros((num_items, bert_embedding_dim))

for item_enc_id, vec_idx in item_enc_to_vec_idx.items():
    desc_embedding_matrix[item_enc_id] = vectors_bert[vec_idx]

desc_embedding_tensor = torch.tensor(desc_embedding_matrix, dtype=torch.float)
print(colored(f'Description embedding matrix shape: {desc_embedding_tensor.shape}', 'blue'))

class HybridNCF(pl.LightningModule):
    def __init__(self, num_users, num_items, ratings_df, all_item_ids,
                 desc_embedding_matrix, user_embedding_dim=16, item_embedding_dim=16,
                 hidden_dim=64, num_negatives=4, lr=1e-3):
        super().__init__()
        self.save_hyperparameters(ignore=['ratings_df', 'all_item_ids', 'desc_embedding_matrix'])
        self.user_embedding = nn.Embedding(num_users, user_embedding_dim)
        self.item_embedding = nn.Embedding(num_items, item_embedding_dim)
        self.desc_embedding = nn.Embedding.from_pretrained(desc_embedding_matrix, freeze=True)
        input_dim = user_embedding_dim + item_embedding_dim + desc_embedding_matrix.shape[1]
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, 32)
        self.output = nn.Linear(32, 1)
        self.ratings_df = ratings_df
        self.all_item_ids = all_item_ids
        self.num_negatives = num_negatives
        self.lr = lr

    def forward(self, user_input, item_input):
        user_emb = self.user_embedding(user_input)
        item_emb = self.item_embedding(item_input)
        desc_emb = self.desc_embedding(item_input)
        x = torch.cat([user_emb, item_emb, desc_emb], dim=-1)
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return torch.sigmoid(self.output(x))

    def training_step(self, batch, batch_idx):
        u, i, labels = batch
        preds = self(u, i).view(-1)
        loss = nn.BCELoss()(preds, labels)
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.lr)

    def train_dataloader(self):
        dataset = ImplicitTrainDataset(self.ratings_df, self.all_item_ids, self.num_negatives)
        return DataLoader(dataset, batch_size=512, shuffle=True, num_workers=0)

print(colored('Training Hybrid NCF...', 'blue'))
model_hybrid = HybridNCF(
    num_users=num_users,
    num_items=num_items,
    ratings_df=train_df,
    all_item_ids=all_item_ids,
    desc_embedding_matrix=desc_embedding_tensor,  # ← CAMBIADO
    user_embedding_dim=16,
    item_embedding_dim=16,
    hidden_dim=64,
    num_negatives=4,
    lr=1e-3
)

trainer2 = pl.Trainer(max_epochs=5, enable_checkpointing=False, logger=False)
trainer2.fit(model_hybrid)
print(colored('Hybrid NCF trained ✓', 'green'))

## 7. Final Comparison

In [ ]:
# Evaluate Hybrid NCF
print(colored('Evaluating Hybrid NCF...', 'blue'))
hr10_hybrid = hit_rate_at_k(model_hybrid, train_user_items_dict, test_user_items_dict, all_item_ids, k=10)
print(colored(f'Hybrid NCF HR@10: {hr10_hybrid:.4f}', 'green'))

# Summary table
results = pd.DataFrame({
    'Model': ['KNN User-based', 'KNN Item-based', 'SVD (Matrix Factorization)',
              'NCF (Deep CF)', 'Hybrid NCF (Deep CF + BERT)'],
    'Type': ['Collaborative', 'Collaborative', 'Collaborative',
             'Deep Collaborative', 'Hybrid Deep'],
    'RMSE': [rmse_knn_user, rmse_knn_item, rmse_svd, '-', '-'],
    'HR@10': [hr10_knn_user, hr10_knn_item, hr10_svd, hr10_ncf, hr10_hybrid],
    'Best params': [
        f"k={best_params_user['k']}, sim={best_params_user['sim_options']['name']}",
        f"k={best_params_item['k']}, sim={best_params_item['sim_options']['name']}",
        'n_factors=20, n_epochs=10, lr=0.005',
        'user_emb=16, item_emb=16, hidden=64',
        'NCF + BERT(384) frozen'
    ]
})
print(colored('\n============= Final Results =============', 'blue'))
print(results.to_string(index=False))

In [ ]:
# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# RMSE comparison (lower is better)
rmse_models = ['KNN User-based', 'KNN Item-based', 'SVD']
rmse_scores = [rmse_knn_user, rmse_knn_item, rmse_svd]
axes[0].bar(rmse_models, rmse_scores, color=['steelblue', 'steelblue', 'coral'])
axes[0].set_title('Collaborative Filtering — RMSE (lower is better)')
axes[0].set_ylabel('RMSE')
axes[0].set_ylim(0, max(rmse_scores) * 1.2)
for i, v in enumerate(rmse_scores):
    axes[0].text(i, v + 0.02, f'{v:.4f}', ha='center', fontsize=10)

# HR@10 comparison (higher is better)
hr_models = ['NCF', 'Hybrid NCF']
hr_scores = [hr10_ncf, hr10_hybrid]
axes[1].bar(hr_models, hr_scores, color=['steelblue', 'coral'])
axes[1].set_title('Deep Models — HR@10 (higher is better)')
axes[1].set_ylabel('Hit Rate @ 10')
axes[1].set_ylim(0, max(hr_scores) * 1.3)
for i, v in enumerate(hr_scores):
    axes[1].text(i, v + 0.005, f'{v:.4f}', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

## 8. Feature Extraction Analysis (PCA on BERT vectors)

In [ ]:
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

# Apply PCA to BERT vectors and analyze impact on content-based recommendations
pca_dims = [10, 25, 50, 100, 200]
test_game = 'com.supercell.clashofclans'
idx = app_id_to_idx[test_game]

print(colored('PCA dimensionality reduction on BERT vectors', 'blue'))
print(colored(f'Original BERT dimensions: {vectors_bert.shape[1]}', 'blue'))
print()

pca_results = []

for n_components in pca_dims:
    pca = PCA(n_components=n_components, random_state=42)
    bert_pca = pca.fit_transform(vectors_bert)
    variance_explained = pca.explained_variance_ratio_.sum()
    
    # Content-based similarity with reduced vectors
    sim_scores = cosine_similarity([bert_pca[idx]], bert_pca)[0]
    top5_indices = sim_scores.argsort()[::-1][1:6]
    top5_titles = df_games.iloc[top5_indices]['title'].tolist()
    
    pca_results.append({
        'n_components': n_components,
        'variance_explained': variance_explained,
        'top1_recommendation': top5_titles[0]
    })
    
    print(colored(f'PCA({n_components:3d} dims) | Variance explained: {variance_explained:.3f} | Top-1: {top5_titles[0]}', 'green'))

df_pca_results = pd.DataFrame(pca_results)

In [ ]:
# Plot variance explained vs dimensions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(pca_dims, [r['variance_explained'] for r in pca_results], 'o-', color='steelblue', linewidth=2)
axes[0].axhline(y=0.95, color='coral', linestyle='--', label='95% variance')
axes[0].axhline(y=0.90, color='green', linestyle='--', label='90% variance')
axes[0].set_title('PCA: Variance Explained vs Number of Components')
axes[0].set_xlabel('Number of Components')
axes[0].set_ylabel('Cumulative Variance Explained')
axes[0].set_xticks(pca_dims)
axes[0].legend()

# Compare content-based RMSE with full BERT vs PCA-100 BERT
# Build PCA-100 item matrix for NCF comparison
pca_optimal = PCA(n_components=100, random_state=42)
bert_pca_100 = pca_optimal.fit_transform(vectors_bert)

# Rebuild item-to-vector mapping with PCA vectors
pca_embedding_matrix = np.zeros((num_items, 100))
for item_enc_id, vec_idx in item_enc_to_vec_idx.items():
    pca_embedding_matrix[item_enc_id] = bert_pca_100[vec_idx]

# Compare avg within-genre similarity: full BERT vs PCA-100
sim_full = cosine_similarity(vectors_bert)
sim_pca = cosine_similarity(bert_pca_100)

genres = df_games['genre'].values
within_full, within_pca = [], []
across_full, across_pca = [], []

for i in range(len(df_games)):
    for j in range(i+1, len(df_games)):
        if genres[i] == genres[j]:
            within_full.append(sim_full[i,j])
            within_pca.append(sim_pca[i,j])
        else:
            across_full.append(sim_full[i,j])
            across_pca.append(sim_pca[i,j])

categories = ['Within-genre\n(BERT full)', 'Within-genre\n(PCA-100)', 
              'Across-genre\n(BERT full)', 'Across-genre\n(PCA-100)']
values = [np.mean(within_full), np.mean(within_pca),
          np.mean(across_full), np.mean(across_pca)]
colors = ['steelblue', 'steelblue', 'coral', 'coral']
alphas = [1.0, 0.5, 1.0, 0.5]

bars = axes[1].bar(categories, values, color=colors)
for bar, alpha in zip(bars, alphas):
    bar.set_alpha(alpha)
axes[1].set_title('Avg Cosine Similarity: Full BERT vs PCA-100')
axes[1].set_ylabel('Average Cosine Similarity')
for i, v in enumerate(values):
    axes[1].text(i, v + 0.003, f'{v:.4f}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

print(colored(f'\nWithin-genre similarity — BERT full: {np.mean(within_full):.4f} | PCA-100: {np.mean(within_pca):.4f}', 'green'))
print(colored(f'Across-genre similarity — BERT full: {np.mean(across_full):.4f} | PCA-100: {np.mean(across_pca):.4f}', 'green'))
print(colored(f'\nConclusion: PCA-100 retains {pca_optimal.explained_variance_ratio_.sum():.1%} variance with {100} dims (vs 384)', 'blue'))

## 9. Hyperparameter Validation (SVD Cross-Validation)

In [ ]:
from surprise.model_selection import cross_validate, GridSearchCV as SurpriseGridSearch

# Grid search for SVD hyperparameters
print(colored('Running SVD cross-validation...', 'blue'))

param_grid = {
    'n_factors': [20, 50, 100],
    'n_epochs': [10, 20],
    'lr_all': [0.005, 0.01],
}

gs = SurpriseGridSearch(SVD, param_grid, measures=['rmse'], cv=3, n_jobs=-1)
gs.fit(data)

print(colored(f'\nBest RMSE: {gs.best_score["rmse"]:.4f}', 'green'))
print(colored(f'Best params: {gs.best_params["rmse"]}', 'green'))

# Show results table
results_df = pd.DataFrame(gs.cv_results)
results_df = results_df[['param_n_factors', 'param_n_epochs', 'param_lr_all', 'mean_test_rmse']].sort_values('mean_test_rmse')
print(colored('\nTop 5 configurations:', 'blue'))
print(results_df.head().to_string(index=False))

In [ ]:
# Retrain best SVD model with optimal params
best_params = gs.best_params['rmse']
svd_best = SVD(
    n_factors=best_params['n_factors'],
    n_epochs=best_params['n_epochs'],
    lr_all=best_params['lr_all'],
    random_state=42
)
svd_best.fit(trainset)
preds_svd_best = svd_best.test(testset)
rmse_svd_best = accuracy.rmse(preds_svd_best, verbose=False)

print(colored(f'SVD default RMSE:   {rmse_svd:.4f}', 'blue'))
print(colored(f'SVD optimized RMSE: {rmse_svd_best:.4f}', 'green'))
print(colored(f'Improvement: {((rmse_svd - rmse_svd_best)/rmse_svd)*100:.2f}%', 'green'))

## 10. Generating Recommendations for a User

In [ ]:
def recommend_for_user(player_id_raw, model, top_k=10):
    """
    Generate top-k game recommendations for a given player using Hybrid NCF.
    Shows games the player has not interacted with yet.
    """
    if player_id_raw not in enc_players.classes_:
        print(colored(f'Player not found: {player_id_raw}', 'red'))
        return None

    user_enc = enc_players.transform([player_id_raw])[0]
    seen_items = train_user_items_dict.get(user_enc, set())
    candidates = [i for i in all_item_ids if i not in seen_items]

    model.eval()
    user_tensor = torch.tensor([user_enc] * len(candidates), dtype=torch.long)
    item_tensor = torch.tensor(candidates, dtype=torch.long)

    with torch.no_grad():
        scores = model(user_tensor, item_tensor).view(-1).cpu().numpy()

    top_idx = np.argsort(scores)[::-1][:top_k]
    top_item_encs = np.array(candidates)[top_idx]
    top_app_ids = enc_items.inverse_transform(top_item_encs)

    recs = df_games[df_games['app_id'].isin(top_app_ids)][['title', 'genre', 'rating']].copy()
    recs['relevance_score'] = scores[top_idx]
    recs = recs.reset_index(drop=True)
    return recs

print(colored('recommend_for_user() defined ✓', 'green'))

In [ ]:
# Pick an example user and show their history + recommendations
example_user = df_interactions['player_id'].value_counts().index[0]

# Games the user has already rated
user_history = df_interactions[df_interactions['player_id'] == example_user][['app_id', 'rating']]
user_history = user_history.merge(df_games[['app_id', 'title', 'genre']], on='app_id', how='left')

print(colored(f'User: {example_user}', 'blue'))
print(colored(f'Games already rated ({len(user_history)}):', 'blue'))
print(user_history[['title', 'genre', 'rating']].to_string(index=False))

print()
print(colored('Hybrid NCF Recommendations:', 'blue'))
recs = recommend_for_user(example_user, model_hybrid, top_k=10)
if recs is not None:
    print(recs.to_string(index=False))

## 11. Summary

In [ ]:
print(colored('============= Final Recommender System Summary =============', 'blue'))
print()
print(colored('Content-Based Filtering:', 'red'))
print('  Cosine similarity using TF-IDF, Word2Vec, LDA and BERT vectors')
print('  PCA analysis: 100 components retain 90.9% BERT variance')
print('  Finding: full BERT (384 dims) outperforms PCA reduction for content-based')
print()
print(colored('Collaborative Filtering:', 'red'))
print(f'  KNN User-based  RMSE: {rmse_knn_user:.4f} | HR@10: {hr10_knn_user:.4f}  (k={best_params_user["k"]}, sim={best_params_user["sim_options"]["name"]})')
print(f'  KNN Item-based  RMSE: {rmse_knn_item:.4f} | HR@10: {hr10_knn_item:.4f}  (k={best_params_item["k"]}, sim={best_params_item["sim_options"]["name"]})')
print(f'  SVD (default)   RMSE: {rmse_svd:.4f}')
print(f'  SVD (optimized) RMSE: {rmse_svd_best:.4f} | HR@10: {hr10_svd:.4f}  ← best params: {best_params}')
print()
print(colored('Deep Recommender Systems (implicit feedback):', 'red'))
print(f'  NCF             HR@10: {hr10_ncf:.4f}')
print(f'  Hybrid NCF+BERT HR@10: {hr10_hybrid:.4f}')
print()
print(colored('Key findings:', 'green'))
print(f'  1. SVD outperforms KNN in explicit rating prediction (RMSE {rmse_svd_best:.4f} vs {rmse_knn_user:.4f})')
print(f'  2. NCF HR@10: {hr10_ncf:.4f} | Hybrid NCF HR@10: {hr10_hybrid:.4f}')
print(f'     Note: deep model results vary across runs due to random initialization')
print(f'  3. PCA reduces BERT from 384 to 100 dims preserving 90.9% variance')
print(f'     but cosine similarity drops — full BERT preferred for content-based')